# Plant Disease Classifier

## Exploratory Data Analysis (EDA)

How many classes are there? What are they? And the size of the images and are they all the same?.

In [ ]:
import os
path = "data/PlantVillage"
classes = os.listdir(path + "/train")
print(len(classes)-1)

In [ ]:
# Number of images in each class
images_per_class = {}
for class_name in classes:
    class_path = os.path.join(path, "train", class_name)
    if os.path.isdir(class_path):
        image_files = os.listdir(class_path)
        images_per_class[class_name] = image_files

In [ ]:
# Plot the number of images in each class
import matplotlib.pyplot as plt
images_counts = []
for k in images_per_class.keys():
    images_counts.append(len(images_per_class[k]))
plt.figure(figsize=(10, 5))
plt.bar(images_per_class.keys(), images_counts)
plt.xlabel("Classes")
plt.xticks(rotation=90)
plt.ylabel("Number of Images")
plt.title("Number of Images in Each Class")
plt.show()

In [ ]:
all_images = [
    os.path.join(path, "train", class_name, filename)
    for class_name, filenames in images_per_class.items()
    for filename in filenames
]
len(all_images)

In [ ]:
from PIL import Image
from collections import Counter

shapes = [Image.open(p).size for p in all_images]  # (width, height)
print(Counter(shapes))
print("all same:", len(set(shapes)) == 1)

Plot the an image of each class

In [ ]:
# Plot an image from each class
import math

n_classes = len(images_per_class)
cols = 5
rows = math.ceil(n_classes / cols)

fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3.2))
axes = axes.flatten()

for i, (key, value) in enumerate(images_per_class.items()):
    img = Image.open(os.path.join(path, "train", key, value[0]))
    axes[i].imshow(img)
    axes[i].set_title(key.replace("___", "\n"), fontsize=8)
    axes[i].axis("off")

# Hide any unused subplots
for ax in axes[n_classes:]:
    ax.axis("off")

plt.tight_layout()
plt.show()

Build the dataset

In [ ]:
import tensorflow as tf
TRAIN_DIR = "data/PlantVillage/train"
VAL_DIR = "data/PlantVillage/val"

IMG_SIZE = (128, 128)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
  TRAIN_DIR,
  image_size=IMG_SIZE,
  batch_size=BATCH_SIZE,
  label_mode="categorical",
  shuffle=True,
  seed=42
)

val_ds = tf.keras.utils.image_dataset_from_directory(
  VAL_DIR,
  image_size=IMG_SIZE,
  batch_size=BATCH_SIZE,
  label_mode="categorical",
  shuffle=False,
)

class_names = train_ds.class_names
print(len(class_names), class_names[:5])

Normalization (0-255 -> 0-1)

In [ ]:
normalize = tf.keras.layers.Rescaling(1.0/255)
AUTOTUNE = tf.data.AUTOTUNE

def prepare(ds, training=False):
  ds = ds.unbatch()                                                                # unbatch the dataset
  ds = ds.map(lambda x, y: (tf.cast(x, tf.uint8), y), num_parallel_calls=AUTOTUNE) # unit8
  ds = ds.cache()                  
  if training:
    ds = ds.shuffle(1000, seed=42)                                                 # remix every epoch
  ds = ds.batch(BATCH_SIZE)
  ds = ds.map(lambda x, y: (normalize(x), y), num_parallel_calls=AUTOTUNE)
  return ds.prefetch(AUTOTUNE)                                                     # Prepare next batch while training on GPU

train_ds = prepare(train_ds, training=True)
val_ds = prepare(val_ds)

In [ ]:
# Validate the preprocess step
import time
def time_pass(ds, name):
  start = time.time()
  for _ in ds:
    pass
  print(f"{name}: {time.time() - start:.1f}s")
  
time_pass(train_ds, "pass 1 (filling cache)")
time_pass(train_ds, "pass 2 (reading from cache)")

In [ ]:
# Tensorflow version & CPU devices
print("TensorFlow version:", tf.__version__)
print("CPU devices:", tf.config.list_physical_devices())

Baseline CNN

In [ ]:
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.models import Model

inputs = Input(shape=(128,128,3))
conv1 = Conv2D(32, kernel_size=(3,3), activation='relu', padding='same')(inputs)
maxpooling1 = MaxPooling2D(pool_size=(3,3))(conv1)
conv2 = Conv2D(64, kernel_size=(3,3), activation='relu', padding='same')(maxpooling1)
maxpooling2 = MaxPooling2D(pool_size=(3,3))(conv2)
conv3 = Conv2D(128, kernel_size=(3,3), activation='relu', padding='same')(maxpooling2)
maxpooling3 = MaxPooling2D(pool_size=(3,3))(conv3)
x = Flatten()(maxpooling3)
x = Dense(128, activation='relu')(x)
outputs = Dense(len(class_names), activation='softmax')(x)

model = Model(inputs=inputs, outputs=outputs)

model.summary()

In [ ]:
model.compile(
  optimizer='adam', 
  loss='categorical_crossentropy', 
  metrics=['accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
        ]
  )

Sanity Checks Before Training

In [ ]:
import math
loss_prob = math.log(len(class_names))
loss_prob

In [ ]:
images, labels = next(iter(train_ds))
result = model.evaluate(images, labels, return_dict=True, verbose=0)
print(result['loss'])

Try to overfit to one batch of images

In [ ]:
history = model.fit(images, labels, epochs=50, verbose=0)
print(f"Started Loss: {history.history['loss'][0]}")
print(f"Ended Loss: {history.history['loss'][-1]}")

Reconstruct and compile the models

In [ ]:
inputs = Input(shape=(128,128,3))
conv1 = Conv2D(32, kernel_size=(3,3), activation='relu', padding='same')(inputs)
maxpooling1 = MaxPooling2D(pool_size=(3,3))(conv1)
conv2 = Conv2D(64, kernel_size=(3,3), activation='relu', padding='same')(maxpooling1)
maxpooling2 = MaxPooling2D(pool_size=(3,3))(conv2)
conv3 = Conv2D(128, kernel_size=(3,3), activation='relu', padding='same')(maxpooling2)
maxpooling3 = MaxPooling2D(pool_size=(3,3))(conv3)
x = Flatten()(maxpooling3)
x = Dense(128, activation='relu')(x)
outputs = Dense(len(class_names), activation='softmax')(x)

model = Model(inputs=inputs, outputs=outputs)

model.summary()

model.compile(
  optimizer='adam', 
  loss='categorical_crossentropy', 
  metrics=['accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
        ]
  )

In [ ]:
small_train = train_ds.take(100)   # batch 100 = ~3,200 images
small_val   = val_ds.take(30)

history = model.fit(small_train, validation_data=small_val, epochs=2)

Reconstruct the model again

In [ ]:
inputs = Input(shape=(128,128,3))
conv1 = Conv2D(32, kernel_size=(3,3), activation='relu', padding='same')(inputs)
maxpooling1 = MaxPooling2D(pool_size=(3,3))(conv1)
conv2 = Conv2D(64, kernel_size=(3,3), activation='relu', padding='same')(maxpooling1)
maxpooling2 = MaxPooling2D(pool_size=(3,3))(conv2)
conv3 = Conv2D(128, kernel_size=(3,3), activation='relu', padding='same')(maxpooling2)
maxpooling3 = MaxPooling2D(pool_size=(3,3))(conv3)
x = Flatten()(maxpooling3)
x = Dense(128, activation='relu')(x)
outputs = Dense(len(class_names), activation='softmax')(x)

model = Model(inputs=inputs, outputs=outputs)

model.summary()

model.compile(
  optimizer='adam', 
  loss='categorical_crossentropy', 
  metrics=['accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
        ]
  )

In [ ]:
# Recheck the loss
images, labels = next(iter(train_ds))
result = model.evaluate(images, labels, return_dict=True, verbose=0)
print(result['loss'])

Train the model with full dataset

In [ ]:
history_baseline = model.fit(train_ds, validation_data=val_ds, epochs=10)

In [ ]:
# Save the model
import json
model.save("baseline.keras")
with open("history_baseline.json", "w") as f:
  json.dump({k: [float(v) for v in vals]
               for k, vals in history_baseline.history.items()}, f)

Plot the accuracy and loss

In [ ]:
import json
import matplotlib.pyplot as plt

with open("history_baseline.json") as f:
    hist = json.load(f)

epochs = range(1, len(hist["loss"]) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Loss
ax1.plot(epochs, hist["loss"], marker="o", label="Train")
ax1.plot(epochs, hist["val_loss"], marker="o", label="Validation")
ax1.set_title("Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Categorical cross-entropy")
ax1.set_xticks(epochs)
ax1.grid(alpha=0.3)
ax1.legend()

# Accuracy
ax2.plot(epochs, hist["accuracy"], marker="o", label="Train")
ax2.plot(epochs, hist["val_accuracy"], marker="o", label="Validation")
ax2.set_title("Accuracy")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_xticks(epochs)
ax2.grid(alpha=0.3)
ax2.legend()

fig.suptitle("Baseline CNN")
plt.tight_layout()
plt.show()

Per-class evaluation

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

y_true = np.concatenate([np.argmax(y.numpy(), axis=1) for _, y in val_ds])
y_pred = np.argmax(model.predict(val_ds), axis=1)

print(classification_report(y_true, y_pred, target_names=class_names, digits=3))
cm = confusion_matrix(y_true, y_pred)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Colour = fraction of each true class (row-normalised), text = raw counts
cm_norm = cm / cm.sum(axis=1, keepdims=True)
annot = np.where(cm > 0, cm.astype(str), "")
labels = [c.replace("___", " - ").replace("_", " ") for c in class_names]

fig, ax = plt.subplots(figsize=(18, 16))
sns.heatmap(
    cm_norm,
    annot=annot,
    fmt="",
    cmap="Blues",
    vmin=0,
    vmax=1,
    xticklabels=labels,
    yticklabels=labels,
    annot_kws={"size": 7},
    linewidths=0.3,
    linecolor="lightgray",
    square=True,
    cbar_kws={"label": "Fraction of true class (recall)", "shrink": 0.8},
    ax=ax,
)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Baseline CNN")
plt.setp(ax.get_xticklabels(), rotation=90, fontsize=8)
plt.setp(ax.get_yticklabels(), rotation=0, fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

# Rebuild cm so the cell doesn't depend on a possibly modified copy in memory
cm = confusion_matrix(y_true, y_pred)
support = cm.sum(axis=1)  # images per true class, diagonal included

# Zero the diagonal on a copy so only misclassifications remain
off_diag = cm.copy()
np.fill_diagonal(off_diag, 0)

# Indices of the 10 largest off-diagonal cells, largest first
top = np.argsort(off_diag, axis=None)[::-1][:10]
rows, cols = np.unravel_index(top, off_diag.shape)

print(f"{'count':>5}  {'% of true':>9}  {'total':>5}  true -> predicted")
for t, p in zip(rows, cols):
    print(f"{off_diag[t, p]:5d}  {off_diag[t, p] / support[t]:9.1%}  {support[t]:5d}  {class_names[t]} -> {class_names[p]}")



In [ ]:
import json

with open("history_baseline.json") as f:
    hist = json.load(f)

N_EPOCHS = 3
metrics = [k for k in hist if not k.startswith("val_")]

print(f"{'epoch':>5}  {'split':<5}" + "".join(f"{m:>11}" for m in metrics))
for e in range(N_EPOCHS):
    print(f"{e + 1:5d}  {'train':<5}" + "".join(f"{hist[m][e]:11.4f}" for m in metrics))
    print(f"{'':5}  {'val':<5}" + "".join(f"{hist['val_' + m][e]:11.4f}" for m in metrics))


In [ ]:
# Check keras version
print("Built-in Keras version:", tf.keras.__version__)

In [ ]:
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.models import Model

SEED = 42
RUN_NAME = "baseline_v2"   # new name not to delete baseline.keras / history_baseline.json 

def make_datasets(seed=SEED):
  train = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=True,
    seed=seed,
  )
  val = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False,
  )
  return prepare(train, training=True), prepare(val)

def build_model(n_classes):
  inputs = Input(shape=(128, 128, 3))
  x = Conv2D(32, kernel_size=(3, 3), activation='relu', padding='same')(inputs)
  x = MaxPooling2D(pool_size=(3, 3))(x)
  x = Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same')(x)
  x = MaxPooling2D(pool_size=(3, 3))(x)
  x = Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same')(x)
  x = MaxPooling2D(pool_size=(3, 3))(x)
  x = Flatten()(x)
  x = Dense(128, activation='relu')(x)
  outputs = Dense(n_classes, activation='softmax')(x)

  model = Model(inputs=inputs, outputs=outputs)
  return model


In [ ]:
tf.keras.utils.set_random_seed(SEED)   # seed all of Python, NumPy, TensorFlow 

train_ds, val_ds = make_datasets()     # start with new shuffle state
model = build_model(len(class_names))  # new weight init

callbacks = [
  tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1,
  ),
  tf.keras.callbacks.ModelCheckpoint(
    f"{RUN_NAME}.keras",
    monitor="val_loss",
    save_best_only=True,
  ),
]

history = model.fit(
  train_ds,
  validation_data=val_ds,
  epochs=30,               # အများဆုံး; EarlyStopping will stop earlier if no improvement
  callbacks=callbacks,
)

In [ ]:
best_epoch = int(np.argmin(history.history["val_loss"])) + 1

with open(f"history_{RUN_NAME}.json", "w") as f:
  json.dump({k: [float(v) for v in vals]
               for k, vals in history.history.items()}, f)

print("epochs run:", len(history.history["loss"]))
print("best epoch:", best_epoch)
print(model.evaluate(val_ds, return_dict=True, verbose=0))  # must match best epoch val metrics

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def evaluate_per_class(model, ds, title, top_k=10):
  # Labels and predictions (val_ds is unshuffled, so the order matches)
  y_true = np.concatenate([np.argmax(y.numpy(), axis=1) for _, y in ds])
  y_pred = np.argmax(model.predict(ds, verbose=0), axis=1)

  # 1. Classification report
  print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

  # 2. Confusion matrix: colour = row-normalised (recall), text = raw counts
  cm = confusion_matrix(y_true, y_pred)
  support = cm.sum(axis=1)
  cm_norm = cm / support[:, None]
  annot = np.where(cm > 0, cm.astype(str), "")
  labels = [c.replace("___", " - ").replace("_", " ") for c in class_names]

  fig, ax = plt.subplots(figsize=(18, 16))
  sns.heatmap(
    cm_norm,
    annot=annot,
    fmt="",
    cmap="Blues",
    vmin=0,
    vmax=1,
    xticklabels=labels,
    yticklabels=labels,
    annot_kws={"size": 7},
    linewidths=0.3,
    linecolor="lightgray",
    square=True,
    cbar_kws={"label": "Fraction of true class (recall)", "shrink": 0.8},
    ax=ax,
  )
  ax.set_xlabel("Predicted")
  ax.set_ylabel("True")
  ax.set_title(title)
  plt.setp(ax.get_xticklabels(), rotation=90, fontsize=8)
  plt.setp(ax.get_yticklabels(), rotation=0, fontsize=8)
  plt.tight_layout()
  plt.show()

  # 3. Top-k most frequent misclassifications (off-diagonal cells)
  off_diag = cm.copy()
  np.fill_diagonal(off_diag, 0)
  top = np.argsort(off_diag, axis=None)[::-1][:top_k]
  rows, cols = np.unravel_index(top, off_diag.shape)

  print(f"\nTop {top_k} confusions")
  print(f"{'count':>5}  {'% of true':>9}  {'total':>5}  true -> predicted")
  for t, p in zip(rows, cols):
    if off_diag[t, p] == 0:
      break
    print(f"{off_diag[t, p]:5d}  {off_diag[t, p] / support[t]:9.1%}  {support[t]:5d}  "
          f"{class_names[t]} -> {class_names[p]}")

  return cm

cm_v2 = evaluate_per_class(model, val_ds, "Baseline CNN v2")

In [ ]:
import json

def log_history(filename, n_epochs=None):
  with open(filename) as f:
    hist = json.load(f)

  metrics = [k for k in hist if not k.startswith("val_")]
  total = len(hist["loss"])
  n_epochs = total if n_epochs is None else min(n_epochs, total)

  print(f"{filename}  ({total} epochs)")
  print(f"{'epoch':>5}  {'split':<5}" + "".join(f"{m:>11}" for m in metrics))
  for e in range(n_epochs):
    print(f"{e + 1:5d}  {'train':<5}" + "".join(f"{hist[m][e]:11.4f}" for m in metrics))
    if f"val_{metrics[0]}" in hist:
      print(f"{'':5}  {'val':<5}" + "".join(f"{hist['val_' + m][e]:11.4f}" for m in metrics))

  return hist


In [ ]:
hist = log_history("history_baseline_v2.json", 3) # first 3 epochs only

Add BatchNormalization

In [ ]:
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, BatchNormalization, Activation
from tensorflow.keras.models import Model

def build_bn_model(n_classes):
  inputs = Input(shape=(128, 128, 3))

  x = Conv2D(32, kernel_size=(3, 3), padding='same', use_bias=False)(inputs)
  x = BatchNormalization()(x)
  x = Activation('relu')(x)
  x = MaxPooling2D(pool_size=(3, 3))(x)

  x = Conv2D(64, kernel_size=(3, 3), padding='same', use_bias=False)(x)
  x = BatchNormalization()(x)
  x = Activation('relu')(x)
  x = MaxPooling2D(pool_size=(3, 3))(x)

  x = Conv2D(128, kernel_size=(3, 3), padding='same', use_bias=False)(x)
  x = BatchNormalization()(x)
  x = Activation('relu')(x)
  x = MaxPooling2D(pool_size=(3, 3))(x)

  x = Flatten()(x)
  x = Dense(128, activation='relu')(x)
  outputs = Dense(n_classes, activation='softmax')(x)

  model = Model(inputs=inputs, outputs=outputs)
  return model


In [ ]:
import os
import json
import numpy as np
import tensorflow as tf
import time

class EpochTimer(tf.keras.callbacks.Callback):
  # Adds "epoch_time" (seconds, train + validation) to the epoch logs,
  # so it ends up in history, the CSV log and the progress bar
  def on_epoch_begin(self, epoch, logs=None):
    self.start = time.time()

  def on_epoch_end(self, epoch, logs=None):
    logs["epoch_time"] = time.time() - self.start

def run_experiment(run_name, build_fn, epochs=30, patience=5, overwrite=False):
  # 0. Protect existing results from being overwritten
  paths = {
    "model":   f"{run_name}.keras",
    "history": f"history_{run_name}.json",
    "csv":     f"history_{run_name}.csv",
  }
  existing = [p for p in paths.values() if os.path.exists(p)]
  if existing:
    if not overwrite:
      raise FileExistsError(
        f"[{run_name}] files already exist: {', '.join(existing)}. "
        f"Use a new run_name, or pass overwrite=True to replace them."
      )
    print(f"[{run_name}] overwrite=True, removing: {', '.join(existing)}")
    for p in existing:
      os.remove(p)   # so a crashed run can't leave old and new files mixed together

  # 1. Seed Python, NumPy and TensorFlow so runs can be reproduced and compared
  tf.keras.backend.clear_session()  # clear any old models from memory
  tf.keras.utils.set_random_seed(SEED)

  # 2. Fresh datasets (new shuffle state for each run)
  train_ds, val_ds = make_datasets()

  # 3. Build and compile the model (new weight init)
  model = build_fn(len(class_names))
  model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy',
          tf.keras.metrics.Precision(name='precision'),
          tf.keras.metrics.Recall(name='recall')
          ]
    )
  
  # 4. Callbacks
  callbacks = [
    EpochTimer(),   # must be first, so the callbacks after it see "epoch_time"
    tf.keras.callbacks.EarlyStopping(
      monitor="val_loss",
      patience=patience,
      restore_best_weights=True,
      verbose=1,
    ),
    tf.keras.callbacks.ModelCheckpoint(
      paths["model"],
      monitor="val_loss",
      save_best_only=True,
    ),
    tf.keras.callbacks.CSVLogger(paths["csv"]),   # writes a row each epoch, so it survives a crash
  ]

  # 5. Fit
  history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs,           # EarlyStopping will stop earlier if no improvement
    callbacks=callbacks,
  )

  # 6. Save the history as JSON
  with open(paths["history"], "w") as f:
    json.dump({k: [float(v) for v in vals]
                 for k, vals in history.history.items()}, f)

  best_epoch = int(np.argmin(history.history["val_loss"])) + 1
  print(f"[{run_name}] epochs run: {len(history.history['loss'])}, best epoch: {best_epoch}")

  return model, history

In [ ]:
import os
import json
from datetime import datetime
import numpy as np
import pandas as pd
import tensorflow as tf

# Classes to compare across runs (exact names from class_names).
WATCH_PAIRS = [
  ("corn_cerc_to_nlb", "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot", "Corn_(maize)___Northern_Leaf_Blight"),
  ("peach_to_apple", "Peach___healthy", "Apple___healthy"),
  ("tom_early_to_late", "Tomato___Early_blight", "Tomato___Late_blight")
  ]
RESULTS_CSV = "results.csv"

def evaluate_experiment(run_name, top_k=10, n_worst=5, n_epochs=None):
  # 1. Load the best checkpoint saved by run_experiment
  model = tf.keras.models.load_model(f"{run_name}.keras")
  _, val_ds = make_datasets()

  # 2. Evaluate, and check against the best epoch in the history
  results = model.evaluate(val_ds, return_dict=True, verbose=0)
  with open(f"history_{run_name}.json") as f:
    hist = json.load(f)
  best = int(np.argmin(hist["val_loss"]))
  print(f"[{run_name}] best epoch: {best + 1} / {len(hist['loss'])}")
  print(f"{'metric':<10}{'evaluate':>10}{'history':>10}")
  for m, v in results.items():
    print(f"{m:<10}{v:10.4f}{hist['val_' + m][best]:10.4f}")
  print()

  # 3-5. Classification report, confusion matrix, top-k confusions
  cm = evaluate_per_class(model, val_ds, run_name, top_k=top_k)

  # 6. Lowest Recall
  support = cm.sum(axis=1)
  predicted = cm.sum(axis=0)
  tp = np.diag(cm)
  recall = tp / np.maximum(support, 1)
  precision = tp / np.maximum(predicted, 1)
  f1 = 2 * precision * recall / np.maximum(precision + recall, 1e-12)

  idx = list(np.argsort(recall)[:n_worst])
  print(f"\nWatch list (lowest {n_worst} recall)")

  off_diag = cm.copy()
  np.fill_diagonal(off_diag, 0)
  print(f"{'precision':>9}  {'recall':>6}  {'f1':>6}  {'total':>5}  class -> most confused with")
  for i in idx:
    j = int(np.argmax(off_diag[i]))
    confused = f"{class_names[j]} ({off_diag[i, j]})" if off_diag[i, j] > 0 else "-"
    print(f"{precision[i]:9.3f}  {recall[i]:6.3f}  {f1[i]:6.3f}  {support[i]:5d}  "
          f"{class_names[i]} -> {confused}")
  print()

  # 7. History log
  log_history(f"history_{run_name}.json", n_epochs)

  # 8. Summary row in results.csv (replaces the row if this run_name is already there)
  epoch_times = hist.get("epoch_time", [])   # missing for runs trained before EpochTimer
  row = {
    "run_name": run_name,
    "evaluated_at": datetime.now().strftime("%Y-%m-%d %H:%M"),
    "params": model.count_params(),
    "epochs_run": len(hist["loss"]),
    "best_epoch": best + 1,
    "best_val_loss": hist["val_loss"][best],
    "val_accuracy": results["accuracy"],
    "macro_f1": float(f1.mean()),
    "epoch1_sec": epoch_times[0] if epoch_times else np.nan,                           # includes cache filling
    "epoch_sec": float(np.mean(epoch_times[1:])) if len(epoch_times) > 1 else np.nan,  # mean of epochs 2+
  }
  
  support = cm.sum(axis=1)          # images per TRUE class (row sums)

  watch = {}
  print("\nWatch pairs")
  for label, true_name, pred_name in WATCH_PAIRS:
      t = class_names.index(true_name)   # row
      p = class_names.index(pred_name)   # column
      count = int(cm[t, p])
      pct = 100 * count / support[t]

      watch[f"watch_{label}_pct"] = round(float(pct), 1)
      print(f"{label:<20} {count:>4} / {support[t]:<4} = {pct:5.1f}%")
      
  row.update(watch)
  
  df = pd.DataFrame([row])
  if os.path.exists(RESULTS_CSV):
    old = pd.read_csv(RESULTS_CSV)
    old = old[old["run_name"] != run_name]
    if len(old):
      df = pd.concat([old, df], ignore_index=True)
  df.to_csv(RESULTS_CSV, index=False)
  print(f"\nSaved [{run_name}] to {RESULTS_CSV}")

  return model, results, cm

In [ ]:
model_bn, history_bn = run_experiment("baseline_v3", build_bn_model)
model_bn, results_bn, cm_bn = evaluate_experiment("baseline_v3", n_epochs=3)